# 📈 MTPR (Multi-Timeframe Price Range) Analysis

A price range analysis indicator based on **Donchian Channel** concepts.

- **Theoretical Foundation**: Richard Donchian (1950s)
- **Core Method**: N-period highest high and lowest low price channels

## 1. Import Libraries

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Download Stock Data

In [ ]:
# Download 5 years of data (MTPR uses 500-day rolling)
# Example: Taiwan Semiconductor (2330.TW)
symbol = '2330.TW'
df = yf.download(symbol, period='5y')

# Handle MultiIndex columns
if isinstance(df.columns, pd.MultiIndex):
    df = df.droplevel(1, axis=1)

print(f"Downloaded {len(df)} rows for {symbol}")
df.tail()

## 3. MTPR Calculation Function

Based on Donchian Channel concepts:
- Long-term range: 500 days
- Medium-term range: 250 days
- Short-term range: 90 days

In [ ]:
def calculate_mtpr(df, ema_period=21):
    """
    Calculate MTPR (Multi-Timeframe Price Range)
    Based on Donchian Channel concepts with multiple timeframes
    
    Parameters:
    - df: DataFrame with 'High' and 'Low' columns
    - ema_period: EMA smoothing period (default 21)
    
    Returns:
    - Series: MTPR values
    """
    
    # Handle MultiIndex columns if present
    if isinstance(df.columns, pd.MultiIndex):
        df = df.droplevel(1, axis=1)
    
    # Donchian-style highest highs (smoothed)
    high_500 = df['High'].rolling(500).max().ewm(span=ema_period).mean()
    high_250 = df['High'].rolling(250).max().ewm(span=ema_period).mean()
    high_90 = df['High'].rolling(90).max().ewm(span=ema_period).mean()
    
    # Donchian-style lowest lows (smoothed)
    low_500 = df['Low'].rolling(500).min().ewm(span=ema_period).mean()
    low_250 = df['Low'].rolling(250).min().ewm(span=ema_period).mean()
    low_90 = df['Low'].rolling(90).min().ewm(span=ema_period).mean()
    
    # Calculate weighted price range center
    mtpr = ((low_500 + low_250 + low_90 + high_500 + high_250 + high_90) / 6).ewm(span=ema_period).mean()
    
    return mtpr

## 4. Analyze Stock

In [ ]:
# Calculate MTPR
mtpr = calculate_mtpr(df.copy())

# Get current values
current_price = float(df['Close'].iloc[-1])
mtpr_value = float(mtpr.dropna().iloc[-1])
ratio = current_price / mtpr_value * 100

print(f"📈 {symbol} Analysis")
print(f"   Current Price: {current_price:.2f}")
print(f"   MTPR Value:    {mtpr_value:.2f}")
print(f"   Ratio:         {ratio:.1f}%")
print()

if ratio < 90:
    print("   ✅ Price below 90% of MTPR - potential opportunity")
elif ratio > 110:
    print("   ⚠️ Price above 110% of MTPR - extended")
else:
    print("   ⏳ Price near MTPR - neutral zone")

## 5. Visualization

In [ ]:
# Plot last 252 trading days (1 year)
fig, ax = plt.subplots(figsize=(14, 7))

# Plot price and MTPR
ax.plot(df['Close'].iloc[-252:], label='Close Price', linewidth=1.5)
ax.plot(mtpr.iloc[-252:], label='MTPR', linewidth=2, color='orange')

ax.set_title(f'{symbol} - Price vs MTPR', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Price')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()